# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Japhetthefirst/flyrank_ml_intership/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane**: Lane 2, Refresh / Content Opportunity Scoring
**Why?**

I picked this lane because it matches a real decision a human will actually makes; with limited time, which pages should be reviewed first?
Also the starter data suggests there's enough to work with.

This lane also lets me compare a transparent rule against a learned model, and report what the data shows even if the model doesn't win. Something similar to the content done in the starter notebook.

- **Declining pages are common.** In the 30,000-page starter data, 54.2% of pages are labelled "declining" (trend_direction = "down"). A reviewer can't look at all of them, so the order matters.
- **Visibility matters.** Among pages with 500+ impressions, 59.6% are declining, versus 38.9% among pages with under 100 impressions. This is observed, not causal: it suggests the pages with the most to lose are the ones worth looking at first.
- **A simple rule alone looks thin here.** Only 17 of 30,000 pages are both stale (180+ days since update) and visible (500+ impressions). So my baseline will need more than one signal.

**Provisional label (and its weakness):** the starter label is "trend_direction = down", which is computed from the current window, not a future outcome. I'll treat it as a proxy, and may move to a future-window label later in the course.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Japhetthefirst/flyrank_ml_intership"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Declining rate:", round((df["trend_direction"] == "down").mean() * 100, 1), "%")

hi = df[df["impressions_90d"] >= 500]
lo = df[df["impressions_90d"] < 100]
print("Declining, 500+ impressions:", round((hi["trend_direction"] == "down").mean() * 100, 1), "%")
print("Declining, <100 impressions:", round((lo["trend_direction"] == "down").mean() * 100, 1), "%")

stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
print("Stale AND visible pages:", int(stale_visible.sum()))

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.
Declining rate: 54.2 %
Declining, 500+ impressions: 59.6 %
Declining, <100 impressions: 38.9 %
Stale AND visible pages: 17


## 2. The question: decision, action, cost of a wrong call

**Question:** Among pages that already have search exposure, which ones should a reviewer look at first because they show signs of declining, and can a learned ranking pick them better than a simple written rule?

**Unit of analysis:** one page (one content_id), using its trailing 90-day numbers.

**Decision this informs:** with limited review time, which pages get reviewed first this cycle.

**Output:** a ranked review queue. Each page gets a score, a reason code (why it's near the top, for example "declining with demand" or "low CTR on a visible page"), and a suggested action.

**Actor and action:** a FlyRank content/SEO reviewer checks the top 50 pages in the queue and decides for each: refresh, expand, protect, prune, or monitor.

**Cost of a wrong call:**
- *False alarm* (a page ranks high but isn't really in trouble): wasted reviewer time, and a review slot taken from a page that needed it.
- *Missed page* (a truly declining page ranks low): the decline goes unnoticed longer.

> **Why data or ML can help at all:** more than one signal is involved (visibility, position, CTR, engagement, age, trend), and they overlap. A single rule like "stale and visible" matches only 17 of 30,000 pages here. A learned ranking might combine the signals better, but only if it beats a simple baseline on clients it has never seen. If it doesn't, the plain rule is the better answer, and I'll report that.

> **Not just "train a model":** the model comes after the decision, the person acting on it, the cost of errors, and the metric (Precision@k, scored on held-out clients) are written down.

> A high rank means "review this first". It does not mean a refresh will fix the page. This data can't show that.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df.shape[0], "pages,", df["client_id"].nunique(), "clients")

# Number 1: how common is decline?
print("\n1) Declining rate:", round(df["is_declining"].mean() * 100, 1), "%",
      f"({df['is_declining_label'].sum():,} pages)")

# Number 2: is decline more common among visible pages?
hi = df[df["impressions_90d"] >= 500]
lo = df[df["impressions_90d"] < 100]
print(f"\n2) Declining rate, 500+ impressions ({len(hi):,} pages):", round(hi["is_declining_label"].mean() * 100, 1), "%")
print(f"   Declining rate, <100 impressions ({len(lo):,} pages):", round(lo["is_declining_label"].mean() * 100, 1), "%")

# Number 3: how different are clients from each other?
by_client = df.groupby("client_id")["is_declining_label"].agg(["size", "mean"])
big = by_client[by_client["size"] >= 100]
print(f"\n3) Among {len(big)} clients with 100+ pages, declining rate ranges from",
      round(big["mean"].min() * 100, 1), "% to", round(big["mean"].max() * 100, 1), "%")

30000 pages, 32 clients

1) Declining rate: 54.2 % (16,262 pages)

2) Declining rate, 500+ impressions (16,726 pages): 59.6 %
   Declining rate, <100 impressions (7,994 pages): 38.9 %

3) Among 24 clients with 100+ pages, declining rate ranges from 0.0 % to 93.7 %


**What these numbers say (observed, directional):**

1. **Decline is common.** 54.2% of pages (16,262 of 30,000) are labelled declining, so a reviewer cannot check them all. A simple "pick at random" approach would be right about half the time, and that's the bar any ranking has to clearly beat.
2. **Visible pages decline more often.** 59.6% of pages with 500+ impressions are declining, against 38.9% of pages with under 100. This is an association, not a cause, but it supports reviewing visible pages first.
3. **Clients differ a lot.** Among the 24 clients with 100+ pages, the share of declining pages ranges from 0% to 93.7%. So part of "decline" is about the client, not the page. I'll test on clients the model has never seen, so it can't win just by recognising a client.

**Why this makes Lane 2 worth the next 7 weeks:** the starter pipeline's own run showed a hand-written rule scoring Precision@50 of 0.24 and a random forest 0.74 on held-out clients. I'll need to earn a result like that again on my own definitions, not assume it.

> Still a proxy: the label comes from the current window, which Section 1 already says.

## 4. Careful words: what I can and can't claim

### What my work WILL say

If my ranking beats the baseline on clients it has never seen, that suggests it can help a reviewer decide which pages to look at first. It does not tell them what to change.

### What my work will NEVER say

**What my work will not claim:** My results describe patterns in observed data. They don't show that refreshing a high-ranked page will make it recover, and they say nothing about how Google ranks pages. Nothing I publish will identify a client, domain, URL, or query.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.